# Gather features and compare three selection strategies

**Deliverable:** three separate ranked tables—`FILTER`, `WRAPPER`, and `EMBEDDED`—with selected flags,
rejected-feature reasons, source lineage, patient support and fold selection stability. This notebook
contains every runtime helper. No other notebook or repository module is required to run it.

**Run order:** configuration → accessible-column inventory → frozen TRAIN population → original claims
features → reviewed additional sources → three selection methods → comparison → three Snowflake tables.
Each code cell is preceded by an explanation of its purpose and outputs. Use an approved Databricks
Python environment with the Spark Snowflake connector, pandas, NumPy, SciPy, scikit-learn and matplotlib.
Provide your existing private Snowflake connection options in cell 1; no credentials belong in this file.

## What was recovered

| Input | Recovered contract |
|---|---|
| Original automated features | 42 `RX__`, 490 `DX__`, 496 `PX__` = 1,028 claim-category count features |
| Monthly representation | Twelve complete rows per snapshot; timestep 0 newest, 11 oldest; nonnegative raw counts |
| Saved source family | `DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63_DL_POC_` plus `FEATURE_MAP`, `TENSOR_MONTHLY`, `SNAPSHOTS`, `PATIENT_SPLIT` |
| Frozen population | 23,151 snapshots, 12,447 patients, 1,345 positive snapshot labels |
| Analysis population here | One latest snapshot per **TRAIN** patient; no holdout feature values or scores are read |
| Feature representation here | Default: sum each original count over the twelve saved monthly bins, giving 1,028 tabular candidates |
| Source extension | Inventory every column visible in the configured schemas; load additional values only through explicit reviewed source contracts |

The original raw-code mappings, deduplication and vocabulary-threshold SQL were **not recovered**. We do
not regenerate or claim to reverse-engineer them. The saved counts are the starting point. Original
event/ingestion cutoffs and RESP construction remain upstream audit limitations. This notebook reads
source inputs and the fixed split, never old model checkpoints, predictions or performance results.

The 49 V63 business features are a **reference definition list**, not automatically a subset of these
1,028 categories. Existing `MODEL_DATA` encoding may depend on RESP; its values are excluded by default.
Provider metrics with a future window, undated patterns and model outputs are not silently added.

**Interpretation:** these are retrospective feature-screening experiments. Repeatedly tuning against
these cross-validation results makes them development evidence. Independent later-period confirmation
is needed before claiming a generalization improvement; selection alone cannot prove overfitting is fixed.


## 1. Configure connection, discovery scope and source contracts

**Why:** source scope, feature windows and selection budgets must be explicit before looking at scores.
The original saved 1,028-feature route runs without additional source contracts. Discovery covers all
tables/views visible to the role in the listed schemas, not every account/database automatically.
Add approved databases/schemas to `DISCOVERY_SCOPES` when needed. Metadata visibility does not guarantee
permission to read values. A missing scope is reported rather than described as empty.

`SUM_WINDOWS=(12,)` ranks annual totals. Setting `(3, 6, 12)` creates 3,084 count-window candidates and
distinguishes recent activity, at higher runtime/memory cost. Windows are saved calendar bins ending at
END_DT, not exactly 90/180/365 days. The newest bin may be partial. We do not sum already-log-transformed
values. Median imputation, optional log1p and scaling are fitted later inside each training fold.

**Adding more sources:** first run the inventory cells. For each source you can substantiate, add a
contract below and rerun from this cell. `snapshot_numeric` needs unique exact patient/date rows, an
explicit numeric column list and audited availability/encoding. `event_categories` needs event date,
ingestion/availability timestamp, category, stable event IDs, transaction filters and a verified zero
interpretation. It counts distinct reviewed event IDs per category over twelve calendar bins. It does
not guess a medical vocabulary, historical provider join, categorical snapshot encoder or long-value
pivot. These sources stay visible as pending work in the inventory.

Example contract structure (replace names with **verified inventory names**; do not enable unchanged):

```python
# snapshot_numeric:
# {"table": "DATABASE.SCHEMA.REVIEWED_RAW_SNAPSHOT_VIEW", "kind": "snapshot_numeric",
#  "patient_column": "PATIENT_ID", "date_column": "END_DT", "columns": ["RAW_FEATURE"],
#  "availability_column": "AVAILABLE_AT", "uses_resp_encoding": False,
#  "available_at_cutoff_verified": True, "review_note": "Reference to reviewed grain, SQL and cutoff evidence"}
# event_categories:
# {"table": "DATABASE.SCHEMA.REVIEWED_EVENTS", "kind": "event_categories",
#  "patient_column": "PATIENT_ID", "date_column": "EVENT_DATE", "availability_column": "AVAILABLE_AT",
#  "category_column": "REVIEWED_CATEGORY", "event_id_columns": ["EVENT_ID", "LINE_ID"],
#  "equals_filters": {"TRANSACTION_STATUS": ["PAID"]}, "uses_resp_encoding": False,
#  "available_at_cutoff_verified": True, "no_event_means_zero_verified": True,
#  "review_note": "Reference to reviewed counting, deduplication, observation and availability evidence"}
```

For an exact historical snapshot without an availability column, use `availability_column=None` only
with `historical_snapshot_certified=True` and documented evidence. Missing snapshot matches remain NaN.
Enrollment absence must not be relabeled as zero activity. This notebook does not broadcast snapshots
into a sequence or apply padding.

**Output:** a fresh run identifier and configuration summary. `WRITE_RESULTS=True` creates three new
aggregate feature tables at the end. Existing tables are never overwritten. Intermediate patient-level
feature values remain in the private session and are not displayed or exported.


In [ ]:
import json
import hashlib
import time
import uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt

# Supply private connection options here using your normal secret-backed setup.
# Example variable name only: sf_options = {...}; never save secret values in the notebook.
connection = globals().get('sf_options_dl_poc', globals().get('sf_options'))
if not isinstance(connection, dict):
    raise RuntimeError('Provide your existing private sf_options or sf_options_dl_poc dictionary.')
sf_options_analysis = dict(connection)
sf_options_analysis.update(sfDatabase='DSVC_TAKEDA_TA_PRIVATE', sfSchema='DS_ML')
if 'spark' not in globals():
    raise RuntimeError('Run on Databricks with the approved Spark Snowflake connector.')

SOURCE_ROOT = 'DSVC_TAKEDA_TA_PRIVATE.DS_ML.TAK861_TX_READY_V63'
SAVED_PREFIX = SOURCE_ROOT + '_DL_POC'
TABLES = {name: SAVED_PREFIX + '_' + name for name in
          ('FEATURE_MAP', 'SNAPSHOTS', 'PATIENT_SPLIT', 'TENSOR_MONTHLY')}
DISCOVERY_SCOPES = [
    {'database': 'DSVC_TAKEDA_TA_PRIVATE', 'schemas': ['DS_ML', 'DS_ML_PROD']},
    {'database': 'DSVC_TAKEDA_FULLMAP_PLAID_PROD', 'schemas': ['COHORT_1009719']},
]
EXPECTED_POPULATION = (23151, 12447, 1345)
EXPECTED_GROUPS = {'RX': 42, 'DX': 490, 'PX': 496}
SUM_WINDOWS = (12,)
EXTRA_SOURCE_CONTRACTS = []  # Review inventory first; examples are documented above.
MAX_CANDIDATES = 6000
MAX_EVENT_CATEGORIES = 2000
SELECTION_CONFIG = {
    'seed': 42, 'n_splits': 3,
    'min_nonzero_patients': 10, 'min_observed_patients': 20,
    'max_missing_fraction': 0.95,
    'filter_max_features': 100, 'redundancy_abs_spearman': 0.95,
    'wrapper_shortlist': 100, 'wrapper_max_features': 50, 'wrapper_rfe_step': 0.25,
    'embedded_max_features': 100, 'elasticnet_C': 0.1, 'elasticnet_l1_ratio': 0.5,
    'linear_max_iter': 2000, 'tree_estimators': 128,
    'tree_max_depth': 6, 'tree_min_samples_leaf': 15, 'n_jobs': 2,
    'log1p_nonnegative': True,
}
RUN_ID = 'R' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6].upper()
OUTPUT_PREFIX = SOURCE_ROOT + '_FEATURE_ANALYSIS_' + RUN_ID
WRITE_RESULTS = True

def read_sql(query):
    return (spark.read.format('snowflake').options(**sf_options_analysis)
            .option('query', query).load().toPandas())

print('Run:', RUN_ID)
print('Versions:', {'numpy': np.__version__, 'pandas': pd.__version__,
                    'scipy': scipy.__version__, 'sklearn': sklearn.__version__})
print('Additional source contracts:', len(EXTRA_SOURCE_CONTRACTS))
print('Only TRAIN feature values are loaded; frozen split metadata is audited across all splits.')


## 2. Load source inventory and extraction helpers

**Why:** explicit helpers validate keys, dates, labels, category order and monthly completeness before
scores can hide data errors. SQL joins preserve exact patient/date identity. Saved count extraction is
pushed into Snowflake, returning roughly 8,712 rows × 1,028 columns rather than a full monthly tensor.
Values are summed on the server; no trained model artifact is loaded. All helper code is embedded here.

**Output:** function definitions only. `known_sources()` contains 32 recovered source leads and the
limitations of each. An unqualified dated table is resolved from live metadata rather than assigned
an invented schema. `LATEST` sources do not silently replace the 20260825 vintage.


In [ ]:
"""Metadata-only source discovery, embedded verbatim in the delivery notebook.

A candidate is a column to REVIEW, never automatic permission to join it or use
it as a predictor. Catalog visibility does not prove SELECT access or timing.
"""
import re

import pandas as pd


def known_sources():
    """Recovered source leads, with unresolved qualifications kept unresolved."""
    private = "DSVC_TAKEDA_TA_PRIVATE.DS_ML."
    v63 = private + "TAK861_TX_READY_V63_"
    claims = "DSVC_TAKEDA_FULLMAP_PLAID_PROD.COHORT_1009719."
    rows = [
        (v63 + "DL_POC_FEATURE_MAP", "saved_vocabulary", "VERIFIED_CONTRACT", "FEATURE_INDEX, FEATURE_NAME, FEATURE_COLUMN; aliases F0000..F1027; original category-building SQL not recovered."),
        (v63 + "DL_POC_TENSOR_MONTHLY", "saved_monthly_counts", "VERIFIED_CONTRACT", "PATIENT_ID, END_DT, RESP, TIME_STEP and mapped numeric aliases; 12 steps, 0 newest. Zero activity is not independently verified observation coverage."),
        (v63 + "DL_POC_SNAPSHOTS", "frozen_population", "VERIFIED_CONTRACT", "23,151 patient/date snapshots; 12,447 patients; 1,345 positive snapshot labels."),
        (v63 + "DL_POC_PATIENT_SPLIT", "frozen_split", "VERIFIED_CONTRACT", "PATIENT_ID, END_DT, RESP, SPLIT, SPLIT_CONFIG; do not select features using validation or test."),
        (v63 + "MODEL_DATA", "encoded_snapshot_features", "ENCODING_REVIEW_REQUIRED", "Exact patient/date alignment previously passed. Numeric/Binary business encoding and upstream feature selection may depend on RESP; raw TRAIN-safe provenance is not established."),
        (v63 + "MODEL_TYPE", "configured_feature_list", "METADATA_ONLY", "Read FEATURES at runtime for the configured 49-feature list. Do not substitute manually transcribed names."),
        (v63 + "FINAL_MODEL", "fitted_feature_summary", "METADATA_ONLY", "Feature descriptions/encoding parameters; not raw feature observations."),
        (v63 + "FEATURES_SUMMARY", "feature_summary", "METADATA_ONLY", "Inventory/lineage evidence; not a predictor source."),
        (v63 + "UNIVERSE_W_FEATURES", "long_feature_values", "ENCODING_REVIEW_REQUIRED", "Observed PATIENT_ID, END_DT, RESP, CAT, VALUE, VALUE_B, DATA_TYPE, PNT_CNT, RESP_CNT. Need category-grain uniqueness, raw-value and encoding provenance before pivoting."),
        (v63 + "MANUAL_FEATURES", "engineered_snapshot_candidates", "ALIGNMENT_REVIEW_REQUIRED", "Prior exact-key audit matched 0 of 23,151 snapshot keys. Do not repair with patient-only joins or shifted dates."),
        (v63 + "CUSTOM_PLAN_FTS", "plan_snapshot_candidates", "TIMING_REVIEW_REQUIRED", "Recover exact patient/date grain, formula, cutoff and historical plan data before use."),
        (v63 + "CUSTOM_SPECIALIST_VISITS_FTS", "specialist_snapshot_candidates", "TIMING_REVIEW_REQUIRED", "Observed NEURO_VISIT_RECENCY_PCT, NEURO_VISITS, NUM_NEURO_SPECIALISTS; formulas and as-of lookup provenance require review."),
        (v63 + "CUSTOM_PT_HCP_ATOPEN_METRICS", "patient_provider_metrics", "TIMING_REVIEW_REQUIRED", "Verify source metric vintage and historical window before use."),
        (v63 + "TMP_CUSTOM_HCP_ATOPEN_METRICS", "provider_metrics", "FUTURE_WINDOW_BLOCKED", "NPI and START_DT/END_DT; observed single window 2025-08 to 2026-08 is unsuitable for the historical V63 cutoffs. Do not join regenerated future values."),
        (v63 + "TMP_CUSTOM_HCP_ATOPEN_NOZOLP", "provider_metrics_intermediate", "TIMING_REVIEW_REQUIRED", "Source lead only; do not assume an as-of historical metric exists."),
        (v63 + "TMP_CUSTOM_HCP_ATOPEN_WITHZOLP", "provider_metrics_intermediate", "TIMING_REVIEW_REQUIRED", "Source lead only; do not assume an as-of historical metric exists."),
        (v63 + "SKIPGRAM_SEQ_EXT", "undated_sequence_patterns", "SNAPSHOT_AUDIT_REQUIRED", "Snapshot patterns lack dated historical reconstruction. Require cutoff and encoding audit; never manufacture monthly history."),
        (v63 + "NT1_PT_DX_INFO", "diagnosis_history_candidates", "TIMING_REVIEW_REQUIRED", "FIRST/LAST_NT1/NT2 and NUM_NT1/NT2_DX leads; lifetime summaries can include post-cutoff records."),
        (v63 + "ZC_AT_GENERIC_PLAN_CONTROL", "plan_access_candidates", "TIMING_REVIEW_REQUIRED", "GENERICS_NTNL_PCT, GENERICS_PLAN_PCT; historical formulas unverified."),
        (private + "ENROLLMENT_20260825", "observation_intervals", "VERIFIED_SCHEMA", "PATIENT_ID, START_DATE, END_DATE; use coverage as an explicit availability contract, not a label or an automatic feature."),
        (private + "HCP_LOOKUP", "provider_dimension", "ASOF_REVIEW_REQUIRED", "Correct recovered schema is DS_ML. HCP_NPI joins RENDERING_NPI; current specialties do not prove historical specialty availability."),
        ("MEDICAL_EVENTS_20260825", "dated_medical_claims", "QUALIFY_AT_RUNTIME", "Dated table name supplied; do not guess database/schema. Resolve from accessible metadata; event grain, code mappings, dedup and availability need explicit contracts."),
        ("PHARMACY_EVENTS_20260825", "dated_pharmacy_claims", "QUALIFY_AT_RUNTIME", "Dated table name supplied; resolve database/schema from metadata. Paid/reversed status, dedup and fill availability require explicit contracts."),
        (claims + "MEDICAL_EVENTS_LATEST", "current_medical_claims", "VINTAGE_REVIEW_REQUIRED", "SERVICE_DATE is an event date, not proof of record availability. LATEST need not reproduce the 20260825 extract."),
        (claims + "PHARMACY_EVENTS_LATEST", "current_pharmacy_claims", "VINTAGE_REVIEW_REQUIRED", "FILL_DATE, NDC11 and DAYS_SUPPLY are source leads. Never substitute current claims silently for a frozen vintage."),
        (claims + "PATIENT_DEMOGRAPHICS_LATEST", "demographic_dimension", "ASOF_REVIEW_REQUIRED", "YEAR_OF_BIRTH supports approximate year-based age only after patient-key and historical availability checks."),
        (claims + "PROVIDERS_LATEST", "provider_dimension", "ASOF_REVIEW_REQUIRED", "Current provider specialty/type is not an as-of dimension."),
        (claims + "PLANS_LATEST", "plan_dimension", "ASOF_REVIEW_REQUIRED", "Current insurance group/segment is not an as-of dimension."),
        ("DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID", "code_reference", "REFERENCE_REVIEW_REQUIRED", "Diagnosis/procedure/drug categories, descriptions and hierarchy; mapping multiplicity must be verified before counting."),
        ("DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.DX_PX_RX_PLAID_ANNUAL_CNT", "upstream_category_counts", "REFERENCE_REVIEW_REQUIRED", "Inclusion-lineage lead only; original 1,028-category threshold/filter SQL is not recovered."),
        ("DSVC_TAKEDA_TA_PRIVATE.DS_ML_PROD.ALL_PROCEDURES_SIMPLE", "procedure_reference", "REFERENCE_REVIEW_REQUIRED", "ICD-10-PCS decomposition; not a substitute for ordered visit patterns."),
        ("TAK861.NARCOLEPSY_MARKET_BASKET_CODES", "drug_basket_reference", "QUALIFY_AT_RUNTIME", "Database qualification not verified; resolve live metadata before use."),
    ]
    return pd.DataFrame(rows, columns=["source", "role", "status", "notes"])


# Callable alias lets the notebook display a fresh DataFrame without mutable state.
KNOWN_SOURCES = known_sources


def classify_inventory(columns):
    """Classify each INFORMATION_SCHEMA column without reading patient values.

    The returned candidate flag means eligible for CONTRACT REVIEW only. Every
    row has included_by_default=False. The saved tensor is loaded by its separate
    FEATURE_MAP adapter, never by inferring numeric field semantics here.
    """
    required = ["TABLE_CATALOG", "TABLE_SCHEMA", "TABLE_NAME", "COLUMN_NAME", "DATA_TYPE"]
    missing = set(required) - set(columns.columns)
    if missing:
        raise ValueError("Inventory lacks required metadata columns: " + repr(sorted(missing)))
    out = columns.copy().reset_index(drop=True)
    if out[required].isna().any().any():
        raise ValueError("Inventory contains null table/column/type identifiers.")
    if out.duplicated(required[:4]).any():
        raise ValueError("Inventory contains duplicate fully qualified column keys.")
    table_keys = required[:3]
    schemas = {
        tuple(str(x).upper() for x in key): set(group.COLUMN_NAME.astype(str).str.upper())
        for key, group in out.groupby(table_keys, sort=False)
    }
    numeric = {"NUMBER", "DECIMAL", "NUMERIC", "INT", "INTEGER", "BIGINT", "SMALLINT", "FLOAT", "FLOAT4", "FLOAT8", "DOUBLE", "DOUBLE PRECISION", "REAL", "BOOLEAN"}
    target_names = {"RESP", "TARGET", "LABEL", "OUTCOME", "RESPONSE", "RESP_CNT", "POSITIVE_COUNT", "Y_TRUE"}
    identifier_names = {"PATIENT_ID", "ID", "OMNI_ID", "OMNL_ID", "NPI", "HCP_NPI", "RENDERING_NPI", "PRESCRIBER_NPI", "CLAIM_ID", "EVENT_ID", "PLAN_ID", "MEMBER_ID", "SOURCE_ID", "RUN_ID", "DATASET_ID", "TIME_STEP", "FEATURE_INDEX", "RN", "RND"}
    decisions = []
    for row in out.itertuples(index=False):
        t, c, d = str(row.TABLE_NAME).upper(), str(row.COLUMN_NAME).upper(), str(row.DATA_TYPE).upper().split("(", 1)[0]
        names = schemas[(str(row.TABLE_CATALOG).upper(), str(row.TABLE_SCHEMA).upper(), t)]
        snapshot = {"PATIENT_ID", "END_DT"}.issubset(names)
        events = "PATIENT_ID" in names and bool(names & {"SERVICE_DATE", "FILL_DATE", "EVENT_DATE"})
        candidate, kind, reason, adapter = False, "unclassified", "Requires a verified feature contract.", "manual_contract"
        if c in target_names or c.startswith(("TARGET_", "OUTCOME_", "RESP_")):
            kind, reason, adapter = "target_or_target_summary", "Target and target-derived statistics are not predictor columns.", "exclude"
        elif c in identifier_names or c.endswith("_ID") or c.endswith("_NPI") or c in {"SPLIT", "SPLIT_CONFIG"}:
            kind, reason, adapter = "identifier_or_assignment", "Use for keys, grouping or source audit only.", "exclude"
        elif d.startswith(("DATE", "TIME")) or c.endswith(("_DT", "_DATE", "_TIMESTAMP")):
            kind, reason, adapter = "date_or_availability", "Retain as alignment/cutoff metadata; a reviewed derived recency may be a separate feature.", "date_contract"
        elif any(x in t for x in ("SHAP", "SCORED", "PREDICTION", "EVALUATION", "CHECKPOINT", "_MODEL_RUN_", "FEATURE_ANALYSIS")) or any(x in c for x in ("SHAP", "PREDICT", "Y_PROB", "Y_SCORE", "COEFFICIENT", "IMPORTANCE")) or c in {"SCORE", "PROBABILITY", "PROB", "PNT_CNT"}:
            kind, reason, adapter = "model_output_or_fit_summary", "Model outputs, fitted summaries and population summary fields cannot become raw predictors.", "exclude"
        elif t.endswith(("MODEL_TYPE", "FINAL_MODEL", "FEATURES_SUMMARY", "FEATURE_MAP", "PATIENT_SPLIT")):
            kind, reason, adapter = "configuration_or_summary", "Source lineage/configuration only.", "metadata_only"
        elif t.endswith("TMP_CUSTOM_HCP_ATOPEN_METRICS"):
            kind, reason, adapter = "future_window_blocked", "Observed metric window is after historical V63 cutoffs; a new valid historical source is required.", "blocked"
        elif t.endswith(("MODEL_DATA", "UNIVERSE_W_FEATURES")):
            candidate, kind, reason, adapter = True, "encoded_snapshot_review", "Review raw/encoded value definition, RESP-dependent transformation fit population, exact grain and cutoff before inclusion.", "reviewed_snapshot_or_long_pivot"
        elif "SKIPGRAM" in t:
            candidate, kind, reason, adapter = True, "undated_snapshot_review", "Undated patterns require snapshot cutoff and encoding audit; not a temporal sequence source.", "reviewed_snapshot"
        elif t.endswith("TENSOR_MONTHLY") and re.fullmatch(r"F[0-9]{4}", c):
            candidate, kind, reason, adapter = True, "saved_claim_count", "Recover name/order from FEATURE_MAP and validate complete 12-step nonnegative counts; do not infer alias meaning.", "saved_tensor_feature_map"
        elif events:
            candidate, kind, reason, adapter = True, "event_attribute_review", "Requires event grain, date/availability, code mapping, count/dedup and coverage contract; categorical code fields are included in this inventory.", "reviewed_event_aggregate"
        elif snapshot and d in numeric:
            candidate, kind, reason, adapter = True, "numeric_snapshot_review", "Requires unique PATIENT_ID+END_DT, exact cohort alignment, feature semantics and cutoff/encoding audit.", "reviewed_snapshot"
        elif snapshot:
            candidate, kind, reason, adapter = True, "categorical_snapshot_review", "Requires exact snapshot grain and TRAIN-fitted category encoding; raw strings are not automatically numeric.", "reviewed_snapshot_encoder"
        elif "PATIENT_ID" in names or any(x in t for x in ("LOOKUP", "REFERENCE", "PLAID", "PROVIDERS", "PLANS", "MARKET_BASKET")):
            candidate, kind, reason, adapter = True, "dimension_or_reference_review", "Requires explicit relation keys, uniqueness and historical validity; never join arbitrary fields by patient alone.", "reviewed_dimension"
        decisions.append((candidate, kind, reason, adapter, False))
    additions = pd.DataFrame(decisions, columns=["candidate", "kind", "exclusion_reason", "proposed_adapter", "included_by_default"])
    return pd.concat([out, additions], axis=1)


"""Warehouse helpers embedded verbatim in the self-contained feature notebook.

No warehouse connection is opened when this file is imported. Read functions are
injected so the contracts and SQL can be checked without private data.
"""
import hashlib
import json
import re

import numpy as np
import pandas as pd


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def qi(name):
    require(isinstance(name, str) and bool(name) and '\x00' not in name,
            'Invalid SQL identifier.')
    return '"' + name.replace('"', '""') + '"'


def qtable(name):
    parts = name.split('.')
    require(len(parts) == 3 and all(re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', p) for p in parts),
            'Use a fully qualified DATABASE.SCHEMA.TABLE name.')
    return '.'.join(qi(p) for p in parts)


def literal(value):
    require(isinstance(value, str), 'SQL category literals must be strings.')
    return "'" + value.replace("'", "''") + "'"


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                     separators=(',', ':')).encode()).hexdigest()


def metadata_contract(snapshots, split, expected=None):
    def clean(frame, with_split=False):
        fields = ['PATIENT_ID', 'END_DT', 'RESP'] + (['SPLIT'] if with_split else [])
        out = frame[fields].copy()
        require(not out.empty and not out.isna().any().any(), 'Null/empty snapshot metadata.')
        require(out.PATIENT_ID.map(lambda v: isinstance(v, str) and bool(v)).all(),
                'PATIENT_ID must retain nonempty string identifiers.')
        dates = pd.to_datetime(out.END_DT, errors='raise')
        require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(),
                'Snapshot dates must be dates without time zones or intraday times.')
        out['END_DT'] = dates.dt.strftime('%Y-%m-%d')
        require(out.RESP.isin([0, 1]).all(), 'RESP must be binary before conversion.')
        out['RESP'] = out.RESP.astype(int)
        require(not out.duplicated(['PATIENT_ID', 'END_DT']).any(), 'Duplicate snapshot keys.')
        return out.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    source, frozen = clean(snapshots), clean(split, True)
    require(source.equals(frozen[['PATIENT_ID', 'END_DT', 'RESP']]),
            'Saved split and snapshot population/labels do not match exactly.')
    require(set(frozen.SPLIT) == {'train', 'validation', 'test'}, 'Unexpected split labels.')
    require(frozen.groupby('PATIENT_ID').SPLIT.nunique().max() == 1, 'Patient split leakage.')
    if expected:
        observed = (len(source), source.PATIENT_ID.nunique(), int(source.RESP.sum()))
        require(observed == tuple(expected), f'Frozen population changed: {observed}.')
    # One independent analysis row per patient. This is not the all-snapshot evaluation population.
    train = (frozen.loc[frozen.SPLIT.eq('train')].sort_values(['PATIENT_ID', 'END_DT'])
             .drop_duplicates('PATIENT_ID', keep='last').reset_index(drop=True))
    require(set(train.RESP) == {0, 1}, 'Latest TRAIN snapshots must contain both classes.')
    return frozen, train


def feature_map_contract(mapping, expected_groups=None):
    ordered = mapping.sort_values('FEATURE_INDEX').reset_index(drop=True)
    require(ordered.FEATURE_INDEX.tolist() == list(range(len(ordered))), 'Invalid feature indices.')
    names, aliases = ordered.FEATURE_NAME.tolist(), ordered.FEATURE_COLUMN.tolist()
    require(bool(names) and len(set(names)) == len(names), 'Empty or duplicate feature names.')
    require(all(isinstance(n, str) and re.match(r'^(RX|DX|PX)__.+', n) for n in names),
            'Unexpected original claim category name.')
    require(aliases == [f'F{i:04d}' for i in range(len(names))], 'Feature aliases/order changed.')
    groups = pd.Series([n.split('__', 1)[0] for n in names]).value_counts().to_dict()
    if expected_groups:
        require(groups == expected_groups, f'Original vocabulary changed: {groups}.')
    return names, aliases


def train_cte(split_table):
    # This SQL must agree with metadata_contract: latest TRAIN snapshot per patient.
    return f'''analysis_keys AS (
      SELECT PATIENT_ID, END_DT, RESP FROM {qtable(split_table)}
      WHERE SPLIT = 'train'
      QUALIFY ROW_NUMBER() OVER (PARTITION BY PATIENT_ID ORDER BY END_DT DESC) = 1
    )'''


def inventory_query(database, schemas):
    require(bool(schemas), 'Configure at least one schema.')
    return f'''SELECT TABLE_CATALOG, TABLE_SCHEMA, TABLE_NAME, COLUMN_NAME,
      DATA_TYPE, ORDINAL_POSITION, IS_NULLABLE
      FROM {qi(database)}.INFORMATION_SCHEMA.COLUMNS
      WHERE TABLE_SCHEMA IN ({', '.join(literal(s) for s in schemas)})
      ORDER BY TABLE_SCHEMA, TABLE_NAME, ORDINAL_POSITION'''


def claim_query(tensor_table, split_table, aliases, windows=(12,)):
    require(bool(windows) and len(set(windows)) == len(windows)
            and all(isinstance(w, int) and 1 <= w <= 12 for w in windows), 'Invalid windows.')
    aggregates = []
    for w in windows:
        for a in aliases:
            aggregates.append(f'SUM(IFF(t.TIME_STEP < {w}, t.{qi(a)}, 0)) AS {qi(a + "_M" + str(w))}')
    invalid = ' OR '.join(f't.{qi(a)} IS NULL OR t.{qi(a)} < 0' for a in aliases)
    return f'''WITH {train_cte(split_table)}
    SELECT k.PATIENT_ID, k.END_DT, k.RESP,
      COUNT(t.TIME_STEP) AS N_ROWS, COUNT(DISTINCT t.TIME_STEP) AS N_STEPS,
      MIN(t.TIME_STEP) AS MIN_STEP, MAX(t.TIME_STEP) AS MAX_STEP,
      SUM(IFF(t.TIME_STEP != FLOOR(t.TIME_STEP), 1, 0)) AS BAD_STEPS,
      SUM(IFF(t.RESP IS NULL OR t.RESP != k.RESP, 1, 0)) AS BAD_LABELS,
      SUM(IFF({invalid}, 1, 0)) AS BAD_VALUES,
      {', '.join(aggregates)}
    FROM analysis_keys k LEFT JOIN {qtable(tensor_table)} t
      ON t.PATIENT_ID = k.PATIENT_ID AND t.END_DT = k.END_DT
    GROUP BY k.PATIENT_ID, k.END_DT, k.RESP
    ORDER BY k.PATIENT_ID, k.END_DT'''


def align_rows(frame, train):
    frame = frame.copy()
    require(frame.PATIENT_ID.map(lambda v: isinstance(v, str)).all(), 'Source patient IDs changed type.')
    dates = pd.to_datetime(frame.END_DT, errors='raise')
    require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(), 'Invalid source snapshot date.')
    frame['END_DT'] = dates.dt.strftime('%Y-%m-%d')
    require(not frame.duplicated(['PATIENT_ID', 'END_DT']).any(), 'Source multiplied snapshot rows.')
    require(len(frame) == len(train), 'Source returned extra or missing snapshot rows.')
    keys = train[['PATIENT_ID', 'END_DT']].merge(frame, how='left', on=['PATIENT_ID', 'END_DT'],
                                               validate='one_to_one', indicator=True)
    require(keys['_merge'].eq('both').all(), 'Source did not align with all analysis keys.')
    return keys.drop(columns='_merge')


def assemble_claim_features(frame, train, names, aliases, tensor_table, windows=(12,)):
    data = align_rows(frame, train)
    require(np.array_equal(data.RESP.to_numpy(), train.RESP.to_numpy()), 'Claim labels changed.')
    require(data.N_ROWS.eq(12).all() and data.N_STEPS.eq(12).all()
            and data.MIN_STEP.eq(0).all() and data.MAX_STEP.eq(11).all()
            and data.BAD_STEPS.eq(0).all(), 'Incomplete/duplicate/non-integer monthly rows.')
    require(data.BAD_LABELS.eq(0).all() and data.BAD_VALUES.eq(0).all(),
            'Monthly labels, nulls or negative counts violate the source contract.')
    values, lineage = {}, []
    for w in windows:
        for name, alias in zip(names, aliases):
            feature = f'{name}__SUM_M{w:02d}'
            values[feature] = pd.to_numeric(data[f'{alias}_M{w}'], errors='raise').to_numpy(dtype=float)
            lineage.append({'FEATURE': feature, 'BASE_FEATURE': name, 'SOURCE_TABLE': tensor_table,
                            'SOURCE_COLUMN': alias, 'FAMILY': name.split('__', 1)[0],
                            'AGGREGATION': 'SUM', 'WINDOW_MONTHS': w, 'MISSING_RULE': 'complete saved monthly counts',
                            'LINEAGE_STATUS': 'SAVED_COUNTS_UPSTREAM_SQL_NOT_RECOVERED',
                            'POINT_IN_TIME_STATUS': 'event/ingestion cutoff requires original build audit'})
    X = pd.DataFrame(values)
    require(np.isfinite(X.to_numpy()).all() and X.ge(0).all().all(), 'Nonfinite/negative claim totals.')
    return X, pd.DataFrame(lineage)


def validate_extra_contract(contract, inventory):
    required = {'table', 'kind', 'patient_column', 'date_column', 'review_note', 'availability_column'}
    require(required.issubset(contract), 'Extra source is missing its explicit join/time/lineage contract.')
    require(contract['kind'] in {'snapshot_numeric', 'event_categories'}, 'Unknown adapter.')
    require(isinstance(contract['review_note'], str) and len(contract['review_note'].strip()) >= 20,
            'Record the reviewed source grain, lookback, counting and encoding evidence.')
    table = contract['table']
    qtable(table)
    require(not re.search(r'(SHAP|SCORED|PREDICT|FINAL_MODEL|FEATURES_SUMMARY|MODEL_TYPE|ATOPEN|SKIPGRAM)',
                          table.upper()), 'Model outputs or known unresolved vintages cannot be candidate inputs.')
    require(contract.get('uses_resp_encoding') is False, 'Source must explicitly rule out RESP-fitted encoding.')
    require(contract.get('available_at_cutoff_verified') is True, 'Verify source availability before the cutoff.')
    available = contract['availability_column']
    require(bool(available) or contract.get('historical_snapshot_certified') is True,
            'Use an availability timestamp, or explicitly certify historical snapshot availability.')
    if contract['kind'] == 'event_categories':
        require(bool(available), 'Event sources require a record-availability timestamp.')
        require(contract.get('category_column') and contract.get('event_id_columns'),
                'Event source requires category and stable event-key columns.')
        require(contract.get('no_event_means_zero_verified') is True,
                'Event zero-fill requires reviewed observation/coverage semantics.')
    else:
        require(bool(contract.get('columns')), 'List reviewed numeric snapshot columns explicitly.')
        require(not contract.get('equals_filters'),
                'Snapshot equals_filters are unsupported; use an audited unique snapshot view.')
        require(not any(re.search(r'(^|_)(RESP|TARGET|LABEL|PREDICTION|SCORE|RND|NPI|PATIENT_ID)($|_)', c.upper())
                        for c in contract['columns']), 'Identifier/outcome/score candidate blocked.')
    catalog = inventory.assign(FQN=inventory.TABLE_CATALOG + '.' + inventory.TABLE_SCHEMA + '.' + inventory.TABLE_NAME)
    table_cols = catalog.loc[catalog.FQN.eq(table)]
    columns = set(table_cols.COLUMN_NAME)
    needed = [contract['patient_column'], contract['date_column']]
    if available:
        needed.append(available)
    needed += contract.get('columns', []) + contract.get('event_id_columns', [])
    if contract.get('category_column'):
        needed.append(contract['category_column'])
    needed += list(contract.get('equals_filters', {}))
    require(set(needed).issubset(columns), 'Configured columns not in the accessible inventory: ' + repr(set(needed)-columns))
    # Numeric snapshot adapter preserves raw numeric values; arbitrary text is not coerced to zero.
    if contract['kind'] == 'snapshot_numeric':
        numeric = r'^(NUMBER|DECIMAL|NUMERIC|FLOAT|DOUBLE|REAL|INT|BIGINT|SMALLINT|BOOLEAN)'
        dtypes = table_cols.set_index('COLUMN_NAME').DATA_TYPE
        require(all(re.match(numeric, str(dtypes[c])) for c in contract['columns']),
                'Snapshot adapter accepts numeric/boolean columns only.')
    return contract


def availability_predicate(c, alias='s'):
    if not c['availability_column']:
        return 'TRUE'
    # Snapshot cutoff is a calendar date, inclusive through that day's end.
    return f'{alias}.{qi(c["availability_column"])} < DATEADD(day, 1, k.END_DT)'


def snapshot_extra_query(c, split_table):
    selected = ', '.join(f's.{qi(column)} AS {qi("V" + str(i))}' for i, column in enumerate(c['columns']))
    return f'''WITH {train_cte(split_table)}
    SELECT k.PATIENT_ID, k.END_DT, s.{qi(c['patient_column'])} AS SOURCE_MATCH, {selected}
    FROM analysis_keys k LEFT JOIN {qtable(c['table'])} s
    ON s.{qi(c['patient_column'])} = k.PATIENT_ID
      AND s.{qi(c['date_column'])} = k.END_DT AND {availability_predicate(c)}
    ORDER BY k.PATIENT_ID, k.END_DT'''


def event_extra_query(c, split_table, window_months=12):
    require(isinstance(window_months, int) and 1 <= window_months <= 12, 'Invalid event window.')
    ids = ', '.join(f's.{qi(v)} AS {qi("E" + str(i))}' for i, v in enumerate(c['event_id_columns']))
    nonnull_ids = ' AND '.join(f's.{qi(v)} IS NOT NULL' for v in c['event_id_columns'])
    conditions = []
    for col, vals in c.get('equals_filters', {}).items():
        require(isinstance(vals, list) and bool(vals), 'Event filters require nonempty value lists.')
        conditions.append(f'CAST(s.{qi(col)} AS VARCHAR) IN ({", ".join(literal(str(v)) for v in vals)})')
    filters = ' AND '.join(conditions) or 'TRUE'
    # DISTINCT is on the explicitly reviewed event identity *and category*, not an arbitrary row.
    return f'''WITH {train_cte(split_table)}, observations AS (
    SELECT DISTINCT k.PATIENT_ID, k.END_DT,
      CAST(s.{qi(c['category_column'])} AS VARCHAR) AS CATEGORY, {ids},
      IFF({nonnull_ids}, 0, 1) AS BAD_EVENT_KEY
    FROM analysis_keys k JOIN {qtable(c['table'])} s
      ON s.{qi(c['patient_column'])} = k.PATIENT_ID
      AND s.{qi(c['date_column'])} >= DATEADD(month, -{window_months-1}, DATE_TRUNC('month', k.END_DT))
      AND s.{qi(c['date_column'])} < DATEADD(day, 1, k.END_DT)
      AND {availability_predicate(c)}
    WHERE {filters})
    SELECT PATIENT_ID, END_DT, CATEGORY, COUNT(*) AS VALUE,
      SUM(BAD_EVENT_KEY) AS BAD_KEYS FROM observations GROUP BY 1,2,3'''


def gather_extras(read_query, contracts, inventory, train, split_table, max_event_categories=2000):
    blocks, rows, audits = [], [], []
    for position, raw in enumerate(contracts):
        c = validate_extra_contract(raw, inventory)
        prefix = 'EXTRA_' + hashlib.sha256(c['table'].encode()).hexdigest()[:12].upper()
        if c['kind'] == 'snapshot_numeric':
            frame = align_rows(read_query(snapshot_extra_query(c, split_table)), train)
            block = {}
            for i, column in enumerate(c['columns']):
                name = prefix + '__' + column
                block[name] = pd.to_numeric(frame['V' + str(i)], errors='raise').to_numpy(dtype=float)
                rows.append({'FEATURE': name, 'BASE_FEATURE': column, 'SOURCE_TABLE': c['table'],
                             'SOURCE_COLUMN': column, 'FAMILY': 'SNAPSHOT', 'AGGREGATION': 'EXACT_KEY',
                             'WINDOW_MONTHS': None, 'MISSING_RULE': 'retain NaN; train-fold imputation',
                             'LINEAGE_STATUS': c['review_note'], 'POINT_IN_TIME_STATUS': 'reviewed contract'})
            blocks.append(pd.DataFrame(block))
            audits.append({'TABLE': c['table'], 'CANDIDATES': len(block),
                           'MATCHED_PATIENTS': int(frame.SOURCE_MATCH.notna().sum())})
        else:
            frame = read_query(event_extra_query(c, split_table))
            require(not frame.empty, 'Reviewed event source returned no analysis events: ' + c['table'])
            require(frame.BAD_KEYS.eq(0).all(), 'Null event identity: cannot deduplicate safely.')
            require(frame.CATEGORY.notna().all(), 'Null event category requires an explicit mapping policy.')
            counts = pd.to_numeric(frame.VALUE, errors='raise').to_numpy(dtype=float)
            require(np.isfinite(counts).all() and (counts >= 0).all()
                    and np.equal(counts, np.floor(counts)).all(),
                    'Event counts must be finite nonnegative integers, never missing.')
            frame['END_DT'] = pd.to_datetime(frame.END_DT).dt.strftime('%Y-%m-%d')
            require(not frame.duplicated(['PATIENT_ID', 'END_DT', 'CATEGORY']).any(), 'Duplicate event aggregates.')
            keys = set(map(tuple, train[['PATIENT_ID', 'END_DT']].to_numpy()))
            require(set(map(tuple, frame[['PATIENT_ID', 'END_DT']].to_numpy())).issubset(keys), 'Extra source keys.')
            categories = sorted(frame.CATEGORY.unique())
            require(len(categories) <= max_event_categories,
                    'Category vocabulary exceeds limit; supply a reviewed category mapping rather than truncate.')
            wide = frame.pivot(index=['PATIENT_ID', 'END_DT'], columns='CATEGORY', values='VALUE')
            wide = wide.reindex(pd.MultiIndex.from_frame(train[['PATIENT_ID', 'END_DT']])).fillna(0)
            block = {}
            for category in categories:
                name = prefix + '__CAT_' + hashlib.sha256(category.encode()).hexdigest()[:16].upper() + '__SUM_M12'
                block[name] = wide[category].to_numpy(dtype=float)
                rows.append({'FEATURE': name, 'BASE_FEATURE': category, 'SOURCE_TABLE': c['table'],
                             'SOURCE_COLUMN': c['category_column'], 'FAMILY': 'EVENT_CATEGORY',
                             'AGGREGATION': 'COUNT_DISTINCT_REVIEWED_EVENT_ID', 'WINDOW_MONTHS': 12,
                             'MISSING_RULE': 'zero if no events; source coverage explicitly certified',
                             'LINEAGE_STATUS': c['review_note'], 'POINT_IN_TIME_STATUS': 'event and availability dates <= cutoff'})
            blocks.append(pd.DataFrame(block))
            audits.append({'TABLE': c['table'], 'CANDIDATES': len(block),
                           'MATCHED_PATIENTS': int(frame.PATIENT_ID.nunique())})
    X = pd.concat(blocks, axis=1) if blocks else pd.DataFrame(index=train.index)
    require(not X.columns.duplicated().any(), 'Duplicate candidate identity in extra source contracts.')
    require(not np.isinf(X.to_numpy(dtype=float)).any(), 'Infinite extra-source values.')
    return X, pd.DataFrame(rows), pd.DataFrame(audits)


IMPLEMENTATION_SHA256 = "eb90e689872e5b41be6fc37609b9a1d45365478f4decc85eb33f6322f907c24b"
print("Source helpers loaded.")


## 3. Inventory all accessible columns and identify pending source work

**Why:** this is the broad feature-discovery step. It includes numeric and categorical candidates,
metadata, model outputs and reference tables, while clearly separating them. A candidate flag means
**review for inclusion**, not a feature already used in training. IDs, RESP and output scores cannot
become predictors. Catalog classification is conservative guidance, not an automatic proof of safety.

**Outputs:** (1) schema discovery status, (2) recovered table leads with live resolutions,
(3) a summary by table/kind, and (4) full `feature_inventory`. Nonvisible source leads and unsupported
dimensions/long-form/categorical sources remain explicit. No patient records are shown. If a needed
source lies elsewhere, add its database/schema to `DISCOVERY_SCOPES` and rerun. When a database is
accessible but a schema returns no metadata, the result says `NO_VISIBLE_COLUMNS`, not "no features".


In [ ]:
inventory_parts, discovery_status = [], []
for scope in DISCOVERY_SCOPES:
    for schema in scope['schemas']:
        try:
            part = read_sql(inventory_query(scope['database'], [schema]))
            status = 'DISCOVERED' if len(part) else 'NO_VISIBLE_COLUMNS'
            inventory_parts.append(part)
            discovery_status.append({'DATABASE': scope['database'], 'SCHEMA': schema,
                                     'STATUS': status, 'COLUMNS': len(part)})
        except Exception as error:
            # Exception text may contain connection details; keep the UI aggregate-only.
            discovery_status.append({'DATABASE': scope['database'], 'SCHEMA': schema,
                                     'STATUS': 'QUERY_FAILED_' + type(error).__name__, 'COLUMNS': 0})
require(any(len(p) for p in inventory_parts), 'No columns were discovered. Check role and configured scopes.')
inventory = pd.concat(inventory_parts, ignore_index=True).drop_duplicates(
    ['TABLE_CATALOG', 'TABLE_SCHEMA', 'TABLE_NAME', 'COLUMN_NAME'])
feature_inventory = classify_inventory(inventory)
feature_inventory['SOURCE_TABLE'] = (feature_inventory.TABLE_CATALOG + '.' +
                                    feature_inventory.TABLE_SCHEMA + '.' + feature_inventory.TABLE_NAME)
visible_tables = sorted(feature_inventory.SOURCE_TABLE.unique())
recovered_sources = known_sources()
recovered_sources['VISIBLE_MATCHES'] = recovered_sources.source.map(
    lambda source: json.dumps([t for t in visible_tables if t == source or t.endswith('.' + source)]))
table_inventory = (feature_inventory.groupby(['SOURCE_TABLE', 'kind'], dropna=False)
                   .agg(COLUMNS=('COLUMN_NAME', 'size'), REVIEW_CANDIDATES=('candidate', 'sum')).reset_index())
display(pd.DataFrame(discovery_status))
display(recovered_sources)
display(table_inventory)
display(feature_inventory)


## 4. Recover the feature vocabulary and freeze the analysis population

**Why:** name/order alignment prevents one feature being mistaken for another. We check that snapshot
keys/labels equal the saved split, and no patient crosses splits. We then take the latest snapshot per
TRAIN patient **without looking at its label**. Each person contributes one independent analysis row;
this avoids giving frequent visitors more weight and makes stratified folds patient-disjoint.

**Output:** aggregate split counts, analysis patient/positive counts and the RX/DX/PX vocabulary sizes.
Analysis prevalence can differ from the full snapshot prevalence; all AP/lift results below use the
analysis population. Existing VALIDATION and TEST feature values are never loaded. The original
feature-order hash stored with the split must match. Counts alone would not prove identity.


In [ ]:
feature_map = read_sql('SELECT FEATURE_INDEX, FEATURE_NAME, FEATURE_COLUMN FROM ' + qtable(TABLES['FEATURE_MAP']))
base_features, aliases = feature_map_contract(feature_map, EXPECTED_GROUPS)
snapshots = read_sql('SELECT PATIENT_ID, END_DT, RESP FROM ' + qtable(TABLES['SNAPSHOTS']))
saved_split = read_sql('SELECT PATIENT_ID, END_DT, RESP, SPLIT, SPLIT_CONFIG FROM ' + qtable(TABLES['PATIENT_SPLIT']))
frozen_metadata, train_metadata = metadata_contract(snapshots, saved_split, EXPECTED_POPULATION)
require(saved_split.SPLIT_CONFIG.nunique() == 1, 'Inconsistent frozen split configuration.')
split_contract = json.loads(saved_split.SPLIT_CONFIG.iloc[0])
original_feature_hash = hashlib.sha256(json.dumps(base_features, ensure_ascii=False).encode()).hexdigest()
require(split_contract.get('feature_order_sha256') == original_feature_hash
        and split_contract.get('n_timesteps') == 12, 'Feature mapping differs from the frozen split.')
display(frozen_metadata.groupby('SPLIT').agg(
    SNAPSHOTS=('RESP', 'size'), PATIENTS=('PATIENT_ID', 'nunique'), POSITIVES=('RESP', 'sum')).reset_index())
display(pd.DataFrame([{'ANALYSIS_PATIENTS': len(train_metadata),
                       'POSITIVE_PATIENTS': int(train_metadata.RESP.sum()),
                       'PREVALENCE': float(train_metadata.RESP.mean()),
                       'REPRESENTATION': 'one latest snapshot per TRAIN patient'}]))
display(pd.Series([name.split('__', 1)[0] for name in base_features]).value_counts().rename('FEATURES').to_frame())


## 5. Gather the original 1,028 claim-category features

**Why:** the saved `FEATURE_MAP` resolves aliases `F0000…F1027`. For each latest TRAIN snapshot,
sum the original raw counts over the configured bins. The query audits twelve distinct steps,
step bounds, labels, nulls and negative counts. It fails on missing or duplicate months; it never
manufactures missing rows, pads inputs or forward-fills values.

**Output:** shape, category counts and `feature_lineage`. `RX__category__SUM_M12` means the sum of that
saved category across timesteps 0–11. This simple first experiment does not preserve visit order,
within-year changes, unique provider identities or exact day-level recency. Use extra windows or a
reviewed event adapter for those hypotheses. Zero counts preserve the source convention; they do not
independently establish insurance coverage. No label-based grouping or outlier removal happens here.


In [ ]:
monthly_table_columns = set(feature_inventory.loc[
    feature_inventory.SOURCE_TABLE.eq(TABLES['TENSOR_MONTHLY']), 'COLUMN_NAME'])
require(monthly_table_columns == set(['PATIENT_ID', 'END_DT', 'RESP', 'TIME_STEP'] + aliases),
        'Monthly source columns do not match the saved feature map.')
require(len(base_features) * len(SUM_WINDOWS) <= MAX_CANDIDATES, 'Configured windows exceed candidate budget.')
claim_aggregates = read_sql(claim_query(TABLES['TENSOR_MONTHLY'], TABLES['PATIENT_SPLIT'], aliases, SUM_WINDOWS))
X_claims, feature_lineage = assemble_claim_features(
    claim_aggregates, train_metadata, base_features, aliases, TABLES['TENSOR_MONTHLY'], SUM_WINDOWS)
del claim_aggregates
print('Claim candidate matrix:', X_claims.shape)
print('Raw matrix memory (MiB):', round(X_claims.memory_usage(deep=True).sum() / 2**20, 1))
display(feature_lineage)


## 6. Gather additional reviewed features and assemble the candidate pool

**Why:** arbitrary joins can multiply claims or attach future values. Snapshot adapters enforce exact
patient/date keys and retain missing values as NaN; event adapters use both event date and availability
timestamp and explicit event-key deduplication. Candidate names include source identity; similarly named
columns from different tables are not assumed to be equivalent. Exact duplicate vectors are handled
later using only each fold's fitting data.

**Outputs:** loaded-source coverage and the final candidate matrix shape. With no extra contracts,
only the original saved claims features are included. The discovered candidate pool is broader than
the loaded pool; that distinction is intentional and shown explicitly. Availability declarations in
the configuration must refer to actual reviewed evidence—they do not verify themselves.


In [ ]:
X_extra, extra_lineage, extra_audit = gather_extras(
    read_sql, EXTRA_SOURCE_CONTRACTS, inventory, train_metadata, TABLES['PATIENT_SPLIT'], MAX_EVENT_CATEGORIES)
X = pd.concat([X_claims, X_extra], axis=1)
feature_lineage = pd.concat([feature_lineage, extra_lineage], ignore_index=True)
require(X.columns.is_unique and len(X.columns) <= MAX_CANDIDATES, 'Duplicate names or candidate budget exceeded.')
require(feature_lineage.FEATURE.tolist() == X.columns.tolist(), 'Candidate lineage order mismatch.')
require(not np.isinf(X.to_numpy(dtype=float)).any(), 'Infinite candidate values.')
y = train_metadata.RESP.to_numpy(dtype=int)
require(train_metadata.PATIENT_ID.is_unique, 'Analysis must have one row per TRAIN patient.')
analysis_fingerprint = fingerprint(train_metadata[['PATIENT_ID', 'END_DT', 'RESP']].to_dict('records'))
value_hash = hashlib.sha256(pd.util.hash_pandas_object(X, index=False).to_numpy().tobytes()).hexdigest()
input_fingerprint = fingerprint({'population': analysis_fingerprint, 'values': value_hash,
                                 'features': list(X.columns)})
display(pd.DataFrame([{'DISCOVERED_COLUMNS': len(feature_inventory),
                       'COLUMNS_REQUIRING_REVIEW': int(feature_inventory.candidate.sum()),
                       'LOADED_CLAIM_CANDIDATES': X_claims.shape[1],
                       'LOADED_EXTRA_CANDIDATES': X_extra.shape[1], 'TOTAL_CANDIDATES': X.shape[1]}]))
if len(extra_audit):
    display(extra_audit)
else:
    print('No additional source contracts configured. Other sources remain inventoried, not loaded.')


## 7. Compare the 49 configured feature names with the claims vocabulary

**Why:** this makes the mapping question explicit without assuming the 49 are a subset of the 1,028.
Read the current V63 `MODEL_TYPE.FEATURES` metadata only. Match exact names to original category names
and discovered columns; no feature values, coefficient summaries or outcome-informed encodings are added.

**Output:** `business_feature_crosswalk`. A name match is a lead, **not a verified formula match**.
Ratios, sequences, age, distinct-provider counts and plan measures need their own definitions. A read
failure is reported and does not invalidate the independent original-count feature analysis.


In [ ]:
business_feature_crosswalk = pd.DataFrame()
try:
    configuration = read_sql('SELECT FEATURES FROM ' + qtable(SOURCE_ROOT + '_MODEL_TYPE'))
    require(len(configuration) == 1, 'Expected one feature configuration row.')
    configured = configuration.FEATURES.iloc[0]
    if isinstance(configured, str):
        configured = json.loads(configured)
    require(isinstance(configured, (list, tuple, np.ndarray)), 'Unrecognized FEATURES encoding.')
    require(all(isinstance(f, str) for f in configured), 'Invalid configured feature names.')
    crosswalk_rows = []
    for name in configured:
        matches = [f for f in base_features if name in (f, f.split('__', 1)[1])]
        locations = feature_inventory.loc[feature_inventory.COLUMN_NAME.eq(name), 'SOURCE_TABLE'].unique().tolist()
        crosswalk_rows.append({'BUSINESS_FEATURE': name, 'CLAIM_NAME_MATCHES': json.dumps(matches),
                               'COLUMN_LOCATIONS': json.dumps(locations),
                               'STATUS': 'NAME_MATCH_DEFINITION_UNVERIFIED' if matches else 'NO_EXACT_CLAIM_NAME_MATCH',
                               'INCLUDED_FROM_MODEL_DATA': False})
    business_feature_crosswalk = pd.DataFrame(crosswalk_rows)
    display(business_feature_crosswalk)
    print('Configured business feature names:', len(configured))
except Exception as error:
    print('Business-name crosswalk unavailable:', type(error).__name__, '; original-count analysis can proceed.')


## 8. Define the three selection methods and TRAIN-fold evaluation

**Why:** rankings must be learned separately from the rows used to assess them. Every fold refits
support/missingness checks, exact-duplicate removal, medians, transformations, scaling and selectors
using its fitting patients only. The final lists are refitted on all TRAIN analysis patients.

| Strategy | Implementation | Read its output as |
|---|---|---|
| Filter | Mutual information (MI) ranking, capped at 100; remove retained-pair absolute Spearman ≥0.95 | Marginal associations after simple quality/redundancy screening |
| Wrapper | MI shortlist of 100, then L2-logistic recursive feature elimination (RFE), down to 50 | Features useful jointly to this bounded linear search; features outside the shortlist were not tested by RFE |
| Embedded | Elastic-net logistic + constrained ExtraTrees; mean of their within-method importance ranks, capped at 100 | An explicit equal-weight ranking heuristic that exposes both component scores |

**L1/L2:** elastic-net logistic uses `elasticnet_C=0.1` (smaller means stronger overall regularization)
and `elasticnet_l1_ratio=0.5` (L1/L2 mixture). Pure L1 is ratio 1. The wrapper and common evaluator use
L2 logistic. These are **fixed initial settings, not tuned optimal values**. Tuning C, mixture or feature
counts later needs an inner training search or separate validation design; do not report the best
repeatedly tried fold score as untouched performance.

MI treats integer-valued features as discrete and other features as continuous; imputation occurs on
the fitting partition. High-cardinality discrete MI can be biased. Spearman and binary-presence phi are
supplementary descriptions. Presence chi-square p-values are exploratory, unadjusted for multiple
testing, and unreliable with small expected cells; they are not selection gates. Weak marginal features
can still matter through interactions, which the constrained tree can sometimes detect. Tree impurity
importance can favor high-cardinality features and divide credit among correlated predictors.

**Output:** function definitions only. No feature is manually assigned priority. Ranks compare features
within a strategy, not numeric score magnitudes across strategies. No patient or high-count outlier is
deleted. Missingness indicators and semantic feature groups are not automatically engineered here.


In [ ]:
"""Self-contained TRAIN-only feature selection; embedded verbatim in the notebook.

Input: one numeric feature row per distinct TRAIN patient, selected upstream by
latest snapshot date without using RESP. No held-out population is accepted here.
Output tables retain rejected columns and explicitly separate eligibility,
selection and method-specific scores. Fold frequencies describe stability, not
probabilities that a feature is causal or clinically important.
"""
import hashlib
import inspect
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, spearmanr
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


SELECTION_DEFAULTS = {
    "seed": 42, "n_splits": 3, "min_nonzero_patients": 10,
    "min_observed_patients": 20, "max_missing_fraction": 0.95,
    "filter_max_features": 100, "redundancy_abs_spearman": 0.95,
    "wrapper_shortlist": 100, "wrapper_max_features": 50,
    "wrapper_rfe_step": 0.25, "embedded_max_features": 100,
    "elasticnet_C": 0.1, "elasticnet_l1_ratio": 0.5,
    "linear_max_iter": 2000, "tree_estimators": 128,
    "tree_max_depth": 6, "tree_min_samples_leaf": 15, "n_jobs": 2,
    "log1p_nonnegative": True,
}


def _selection_config(config):
    incoming = {} if config is None else dict(config)
    unknown = set(incoming) - set(SELECTION_DEFAULTS)
    if unknown:
        raise ValueError(f"Unknown selection settings: {sorted(unknown)}")
    cfg = {**SELECTION_DEFAULTS, **incoming}
    for key in ("n_splits", "min_nonzero_patients", "min_observed_patients",
                "filter_max_features", "wrapper_shortlist", "wrapper_max_features",
                "embedded_max_features", "linear_max_iter", "tree_estimators",
                "tree_max_depth", "tree_min_samples_leaf", "n_jobs"):
        if type(cfg[key]) is not int or cfg[key] < 1:
            raise ValueError(f"{key} must be a positive integer.")
    if cfg["n_splits"] < 2:
        raise ValueError("At least two TRAIN folds are required.")
    for key in ("max_missing_fraction", "redundancy_abs_spearman",
                "wrapper_rfe_step", "elasticnet_l1_ratio"):
        if not 0 < cfg[key] <= 1:
            raise ValueError(f"{key} must be in (0, 1].")
    if cfg["elasticnet_C"] <= 0:
        raise ValueError("elasticnet_C must be positive.")
    if not isinstance(cfg["log1p_nonnegative"], bool):
        raise ValueError("log1p_nonnegative must be boolean.")
    return cfg


def _selection_inputs(X, y, cfg):
    if not isinstance(X, pd.DataFrame) or X.empty or not X.columns.is_unique:
        raise ValueError("X must be a nonempty DataFrame with unique feature names.")
    if not all(isinstance(c, str) and c for c in X.columns):
        raise ValueError("Feature names must be nonempty strings.")
    forbidden = {"RESP", "PATIENT_ID", "END_DT", "SPLIT", "TIME_STEP"}
    if forbidden.intersection(c.upper() for c in X.columns):
        raise ValueError("Remove identifiers, dates, split and target from predictors.")
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X.dtypes):
        raise ValueError("Predictors must be numeric; do not silently encode identifiers.")
    values = X.to_numpy(dtype=np.float64, na_value=np.nan)
    if np.isinf(values).any():
        raise ValueError("Infinite feature values are not supported.")
    labels = np.asarray(y)
    if labels.shape != (len(X),) or not np.isin(labels, [0, 1]).all():
        raise ValueError("y must contain one binary label per feature row.")
    counts = np.bincount(labels.astype(int), minlength=2)
    if counts.min() < cfg["n_splits"]:
        raise ValueError("Each class must have at least n_splits distinct TRAIN patients.")
    return values, labels.astype(np.int64)


def _fit_quality(values, names, cfg):
    """All thresholds, duplicates, imputation and scaling use this fit partition."""
    observed = np.isfinite(values)
    nonzero = (observed & (values != 0)).sum(axis=0)
    n_observed = observed.sum(axis=0)
    missing = 1 - n_observed / len(values)
    audit = pd.DataFrame({"FEATURE_NAME": names,
        "OBSERVED_TRAIN_PATIENTS": n_observed,
        "NONZERO_TRAIN_PATIENTS": nonzero,
        "MISSING_FRACTION": missing, "ELIGIBLE": False,
        "QUALITY_REASON": "eligible", "DUPLICATE_OF": ""})
    kept = []
    hashes = {}
    for j, name in enumerate(names):
        present = values[observed[:, j], j]
        if not len(present):
            reason = "all_missing"
        elif n_observed[j] < cfg["min_observed_patients"]:
            reason = "insufficient_observations"
        elif missing[j] > cfg["max_missing_fraction"]:
            reason = "high_missingness"
        elif np.ptp(present) == 0:
            reason = "constant_observed_values"
        elif nonzero[j] < cfg["min_nonzero_patients"]:
            reason = "insufficient_nonzero_support"
        else:
            # Canonicalize -0 and NaN before hashing; compare to avoid collisions.
            canonical = values[:, j].copy()
            canonical[canonical == 0] = 0.0
            canonical[np.isnan(canonical)] = np.nan
            digest = hashlib.sha256(canonical.tobytes()).hexdigest()
            duplicate = next((k for k in hashes.get(digest, [])
                if np.array_equal(values[:, j], values[:, k], equal_nan=True)), None)
            if duplicate is not None:
                reason = "exact_duplicate"
                audit.loc[j, "DUPLICATE_OF"] = names[duplicate]
            else:
                hashes.setdefault(digest, []).append(j)
                kept.append(j)
                audit.loc[j, "ELIGIBLE"] = True
                reason = "eligible"
        audit.loc[j, "QUALITY_REASON"] = reason
    if not kept:
        return audit, np.array([], dtype=int), None
    indices = np.asarray(kept, dtype=int)
    raw = values[:, indices]
    medians = np.nanmedian(raw, axis=0)
    imputed = np.where(np.isnan(raw), medians, raw)
    nonnegative = np.nanmin(raw, axis=0) >= 0
    discrete = np.array([np.allclose(col[np.isfinite(col)],
        np.rint(col[np.isfinite(col)]), rtol=0, atol=1e-9) for col in raw.T])
    transformed = imputed.copy()
    log_columns = nonnegative & cfg["log1p_nonnegative"]
    transformed[:, log_columns] = np.log1p(transformed[:, log_columns])
    scaler = StandardScaler().fit(transformed)
    state = {"medians": medians, "log_columns": log_columns, "scaler": scaler,
             "raw": imputed, "Z": scaler.transform(transformed), "discrete": discrete}
    return audit, indices, state


def _quality_transform(values, indices, state):
    data = values[:, indices]
    data = np.where(np.isnan(data), state["medians"], data).copy()
    # A negative holdout value cannot be silently passed through a fitted log rule.
    if np.any(data[:, state["log_columns"]] < 0):
        raise ValueError("Negative value conflicts with a TRAIN-fitted nonnegative feature contract.")
    data[:, state["log_columns"]] = np.log1p(data[:, state["log_columns"]])
    return state["scaler"].transform(data)


def _mi_ranking(raw, y, discrete, names, cfg):
    mi = mutual_info_classif(raw, y, discrete_features=discrete,
                            random_state=cfg["seed"])
    mi = np.nan_to_num(mi, nan=0.0)
    order = sorted(range(len(names)), key=lambda j: (-mi[j], names[j]))
    return mi, np.asarray(order, dtype=int)


def _linear(cfg, penalty="l2"):
    # sklearn >=1.8 expresses penalties through l1_ratio; earlier versions
    # require an explicit penalty argument. Keep the delivered notebook portable.
    penalty_parameter = inspect.signature(LogisticRegression).parameters.get("penalty")
    modern_penalty = penalty_parameter is None or penalty_parameter.default == "deprecated"
    if penalty == "elasticnet":
        arguments = {} if modern_penalty else {"penalty": "elasticnet"}
        return LogisticRegression(**arguments, solver="saga",
            C=cfg["elasticnet_C"], l1_ratio=cfg["elasticnet_l1_ratio"],
            max_iter=cfg["linear_max_iter"], class_weight="balanced",
            random_state=cfg["seed"])
    arguments = {"l1_ratio": 0.0} if modern_penalty else {"penalty": "l2"}
    return LogisticRegression(**arguments, solver="lbfgs", C=1.0,
        max_iter=cfg["linear_max_iter"], class_weight="balanced",
        random_state=cfg["seed"])


def _fit_capture(model, X, y):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(X, y)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return model, converged


def _blank_selection(audit, strategy):
    table = audit.copy()
    table["STRATEGY"] = strategy
    table["SELECTED"] = False
    table["RANK"] = np.nan
    table["SELECTION_REASON"] = table["QUALITY_REASON"]
    table.loc[table.ELIGIBLE, "SELECTION_REASON"] = "below_selection_cutoff"
    return table


def _fit_strategies(values, y, names, cfg):
    audit, indices, state = _fit_quality(values, names, cfg)
    observed = np.isfinite(values)
    nonzero = observed & (values != 0)
    for label, label_name in ((1, "POSITIVE"), (0, "NEGATIVE")):
        audit[f"NONZERO_{label_name}_TRAIN_PATIENTS"] = nonzero[y == label].sum(axis=0)
        audit[f"MISSING_{label_name}_TRAIN_PATIENTS"] = (~observed[y == label]).sum(axis=0)
    tables = {s: _blank_selection(audit, s) for s in ("filter", "wrapper", "embedded")}
    status = {s: True for s in tables}
    if not len(indices):
        return tables, indices, state, status
    kept_names = [names[i] for i in indices]
    raw, Z = state["raw"], state["Z"]
    mi, order = _mi_ranking(raw, y, state["discrete"], kept_names, cfg)

    # Filter: MI ranks individual associations; Spearman removes near-duplicates.
    table = tables["filter"]
    for col in ("MUTUAL_INFORMATION", "SPEARMAN_RESP", "PHI_PRESENCE_RESP",
                "CHI2_PRESENCE", "CHI2_P_EXPLORATORY", "CHI2_MIN_EXPECTED_CELL"):
        table[col] = np.nan
    table["REDUNDANT_WITH"] = ""
    table["MI_TREATS_AS_DISCRETE"] = False
    table.loc[indices, "MUTUAL_INFORMATION"] = mi
    table.loc[indices, "MI_TREATS_AS_DISCRETE"] = state["discrete"]
    correlation = pd.DataFrame(raw).corr(method="spearman").to_numpy()
    selected = []
    for rank, j in enumerate(order, 1):
        i = indices[j]
        table.loc[i, "RANK"] = rank
        observed_rows = observed[:, i]
        observed_values, observed_labels = values[observed_rows, i], y[observed_rows]
        # Association diagnostics use observed pairs, never median-filled labels
        # of absence. MI/selection separately uses the fitted numeric imputation.
        if len(np.unique(observed_labels)) == 2:
            table.loc[i, "SPEARMAN_RESP"] = float(spearmanr(observed_values, observed_labels).statistic)
        presence = observed_values != 0
        if len(np.unique(presence)) == 2 and len(np.unique(observed_labels)) == 2:
            table.loc[i, "PHI_PRESENCE_RESP"] = np.corrcoef(presence.astype(float), observed_labels)[0, 1]
            contingency = np.array([[np.sum((presence == a) & (observed_labels == b))
                                      for b in (0, 1)] for a in (False, True)])
            chi, p, _, expected_cells = chi2_contingency(contingency, correction=False)
            table.loc[i, "CHI2_PRESENCE"] = chi
            table.loc[i, "CHI2_P_EXPLORATORY"] = p
            table.loc[i, "CHI2_MIN_EXPECTED_CELL"] = expected_cells.min()
        redundant = next((k for k in selected
            if abs(correlation[j, k]) >= cfg["redundancy_abs_spearman"]), None)
        if redundant is not None:
            table.loc[i, "SELECTION_REASON"] = "spearman_redundancy"
            table.loc[i, "REDUNDANT_WITH"] = kept_names[redundant]
        elif mi[j] <= 0:
            table.loc[i, "SELECTION_REASON"] = "zero_estimated_mutual_information"
        elif len(selected) < cfg["filter_max_features"]:
            selected.append(j)
            table.loc[i, ["SELECTED", "SELECTION_REASON"]] = [True, "selected"]

    # Wrapper: bounded RFE. It is conditional on an inner-fit MI shortlist.
    table = tables["wrapper"]
    table["SHORTLIST_MI"] = np.nan
    table["RFE_RANK"] = np.nan
    table["FINAL_ABS_COEFFICIENT"] = np.nan
    table.loc[indices, "SELECTION_REASON"] = "outside_mi_shortlist"
    shortlist = order[:cfg["wrapper_shortlist"]]
    shortlist_indices = indices[shortlist]
    table.loc[shortlist_indices, "SHORTLIST_MI"] = mi[shortlist]
    n_select = min(cfg["wrapper_max_features"], len(shortlist))
    if len(shortlist) == 1:
        estimator, converged = _fit_capture(_linear(cfg), Z[:, shortlist], y)
        rfe_rank = np.ones(1, dtype=int)
        support = np.ones(1, dtype=bool)
    else:
        rfe, converged = _fit_capture(RFE(_linear(cfg), n_features_to_select=n_select,
                step=cfg["wrapper_rfe_step"]), Z[:, shortlist], y)
        estimator, rfe_rank, support = rfe.estimator_, rfe.ranking_, rfe.support_
    status["wrapper"] = converged
    table.loc[shortlist_indices, "RFE_RANK"] = rfe_rank
    table.loc[shortlist_indices, "SELECTION_REASON"] = "eliminated_by_rfe"
    selected_indices = shortlist_indices[support]
    table.loc[selected_indices, "SELECTED"] = True
    table.loc[selected_indices, "SELECTION_REASON"] = "selected"
    table.loc[selected_indices, "FINAL_ABS_COEFFICIENT"] = np.abs(estimator.coef_[0])
    wrapper_order = sorted(shortlist_indices, key=lambda i: (
        table.loc[i, "RFE_RANK"],
        -np.nan_to_num(table.loc[i, "FINAL_ABS_COEFFICIENT"]), names[i]))
    table.loc[wrapper_order, "RANK"] = np.arange(1, len(wrapper_order) + 1)

    # Embedded: explicit equally weighted rank aggregation of two fitted models.
    table = tables["embedded"]
    elastic, converged = _fit_capture(_linear(cfg, "elasticnet"), Z, y)
    status["embedded"] = converged
    trees = ExtraTreesClassifier(n_estimators=cfg["tree_estimators"],
        max_depth=cfg["tree_max_depth"], min_samples_leaf=cfg["tree_min_samples_leaf"],
        max_features="sqrt", class_weight="balanced", n_jobs=cfg["n_jobs"],
        random_state=cfg["seed"])
    trees.fit(Z, y)
    coefficient = elastic.coef_[0]
    tree_importance = trees.feature_importances_
    coefficient_rank = pd.Series(np.abs(coefficient)).rank(ascending=False, method="average").to_numpy()
    tree_rank = pd.Series(tree_importance).rank(ascending=False, method="average").to_numpy()
    mean_rank = (coefficient_rank + tree_rank) / 2
    for col in ("ELASTICNET_COEFFICIENT", "ELASTICNET_ABS_COEFFICIENT",
                "TREE_IMPURITY_IMPORTANCE", "ELASTICNET_RANK", "TREE_RANK", "MEAN_COMPONENT_RANK"):
        table[col] = np.nan
    table["ELASTICNET_NONZERO"] = False
    table.loc[indices, "ELASTICNET_COEFFICIENT"] = coefficient
    table.loc[indices, "ELASTICNET_ABS_COEFFICIENT"] = np.abs(coefficient)
    table.loc[indices, "ELASTICNET_NONZERO"] = np.abs(coefficient) > 1e-8
    table.loc[indices, "TREE_IMPURITY_IMPORTANCE"] = tree_importance
    table.loc[indices, "ELASTICNET_RANK"] = coefficient_rank
    table.loc[indices, "TREE_RANK"] = tree_rank
    table.loc[indices, "MEAN_COMPONENT_RANK"] = mean_rank
    embedded_order = sorted(range(len(indices)), key=lambda j: (mean_rank[j], kept_names[j]))
    n_selected = 0
    for rank, j in enumerate(embedded_order, 1):
        i = indices[j]
        table.loc[i, "RANK"] = rank
        if abs(coefficient[j]) <= 1e-8 and tree_importance[j] <= 0:
            table.loc[i, "SELECTION_REASON"] = "zero_importance_in_both_models"
        elif n_selected < cfg["embedded_max_features"]:
            table.loc[i, ["SELECTED", "SELECTION_REASON"]] = [True, "selected"]
            n_selected += 1
    return tables, indices, state, status


def _screening_metrics(y, probability):
    prevalence = float(np.mean(y))
    # Fractional inclusion of an entire tied boundary group avoids arbitrary
    # inflated/deflated top-decile results for constant or heavily tied scores.
    target = max(1, int(np.ceil(0.1 * len(y))))
    boundary = np.sort(probability)[-target]
    above, tied = probability > boundary, probability == boundary
    fraction = (target - above.sum()) / tied.sum()
    top_precision = float((y[above].sum() + fraction * y[tied].sum()) / target)
    return {"AVERAGE_PRECISION": float(average_precision_score(y, probability)),
            "POSITIVE_PREVALENCE": prevalence, "TOP10_PRECISION": top_precision,
            "TOP10_LIFT": top_precision / prevalence}


def select_all(X, y, config=None):
    """Return three feature tables, outer TRAIN-fold diagnostics, and settings.

    The upstream loader must enforce one latest snapshot per TRAIN patient.
    TRAIN cross-validation re-fits *all* preprocessing and selectors separately
    inside every fold. These fixed designs are evaluated with the same L2
    logistic classifier; results are screening diagnostics, not Transformer
    performance or an unbiased estimate after tuning against these results.
    Final lists are then refit on all supplied TRAIN patients. No test is read.
    """
    cfg = _selection_config(config)
    values, labels = _selection_inputs(X, y, cfg)
    names = list(X.columns)
    strategies = ("filter", "wrapper", "embedded")
    selected_counts = {s: np.zeros(len(names), dtype=int) for s in strategies}
    eligible_counts = np.zeros(len(names), dtype=int)
    diagnostics = []
    cv = StratifiedKFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["seed"])
    for fold, (fit_rows, score_rows) in enumerate(cv.split(values, labels), 1):
        tables, indices, state, convergence = _fit_strategies(
            values[fit_rows], labels[fit_rows], names, cfg)
        eligible_counts[indices] += 1
        fit_Z = state["Z"] if state is not None else None
        score_Z = _quality_transform(values[score_rows], indices, state) if state is not None else None
        for strategy in (*strategies, "all_eligible"):
            if strategy == "all_eligible":
                mask = np.zeros(len(names), dtype=bool)
                mask[indices] = True
            else:
                mask = tables[strategy].SELECTED.to_numpy(dtype=bool)
                selected_counts[strategy] += mask
            columns = np.flatnonzero(mask[indices])
            scoring_converged = True
            if len(columns):
                model, scoring_converged = _fit_capture(_linear(cfg), fit_Z[:, columns], labels[fit_rows])
                probability = model.predict_proba(score_Z[:, columns])[:, 1]
            else:
                probability = np.repeat(labels[fit_rows].mean(), len(score_rows))
            diagnostics.append({"STRATEGY": strategy, "FOLD": fold,
                "FIT_PATIENTS": len(fit_rows), "SCORE_PATIENTS": len(score_rows),
                "SELECTED_FEATURES": int(mask.sum()),
                "SELECTOR_CONVERGED": convergence.get(strategy, True),
                "SCORER_CONVERGED": scoring_converged,
                "EVALUATION": "TRAIN_outer_fold_fixed_L2_logistic",
                **_screening_metrics(labels[score_rows], probability)})
    final_tables, _, _, final_convergence = _fit_strategies(values, labels, names, cfg)
    for strategy in strategies:
        table = final_tables[strategy]
        table["FOLD_SELECTION_COUNT"] = selected_counts[strategy]
        table["FOLD_SELECTION_FREQUENCY"] = selected_counts[strategy] / cfg["n_splits"]
        table["FOLD_ELIGIBILITY_FREQUENCY"] = eligible_counts / cfg["n_splits"]
        table["FINAL_SELECTOR_CONVERGED"] = final_convergence[strategy]
        table["FIT_POPULATION"] = "latest_snapshot_per_TRAIN_patient"
        final_tables[strategy] = table.sort_values(["SELECTED", "RANK", "FEATURE_NAME"],
            ascending=[False, True, True], na_position="last").reset_index(drop=True)
    return {**final_tables, "diagnostics": pd.DataFrame(diagnostics), "config": cfg}

print("Selection helpers loaded.")


## 9. Run selection and compare against all eligible features

**Why:** compare all three selected subsets with the full quality-eligible feature pool on identical
TRAIN folds and the same L2-logistic evaluator. This isolates feature-list usefulness for a fixed model;
it does not measure Transformer, LightGBM or the client's fitted model performance. The wrapper uses
fewer features by default, so this is not an equal-feature-count contest; change budgets explicitly for
that experiment. Selection and the scorer never see the scoring fold during fit.

**Outputs:** elapsed runtime and fold diagnostics. AP is `average_precision_score`; the no-skill AP
reference is positive prevalence. Top-10% lift = precision in the highest-scoring tenth / fold prevalence.
Boundary ties receive fractional inclusion instead of arbitrary row-order tie-breaking. Fold standard
deviations reflect variation, not confidence intervals. Nonconverged fits must be resolved before using
coefficients as a final feature recommendation. Runtime depends on feature windows and source expansion.


In [ ]:
started = time.perf_counter()
selection_result = select_all(X, y, SELECTION_CONFIG)
diagnostics = selection_result['diagnostics']
display(diagnostics)
print('Selection runtime (minutes):', round((time.perf_counter() - started) / 60, 2))
print('Candidates:', X.shape[1], '| independent TRAIN patients:', X.shape[0])


## 10. Display the three ranked feature tables

**Why:** keep the complete audit instead of discarding rejected columns. Each table has one row per
candidate, `SELECTED`, `RANK`, quality and selection reasons, source table/column, support and fold
stability. Use `SELECTED=True` to obtain that strategy's final list. Unranked/unevaluated columns retain
a null rank and an explicit reason. An arbitrary top-100 is a computation budget, not a proven optimum.

**How to read trends:** high rank with consistent fold selection and broad positive/negative support is
a stronger follow-up candidate than a high rank supported by a few patients. `FOLD_SELECTION_FREQUENCY`
is a stability fraction, **not a probability of correctness**. High redundancy suggests interchangeable
information, not automatically a useless clinical concept. A negative elastic-net coefficient is a
conditional association, not evidence that changing the feature would change the outcome.

The embedded table exposes elastic-net and tree scores separately; a zero L1 coefficient may coexist
with tree importance. RFE coefficients are comparable only after the fitted scaling and conditional on
its selected set. No current score is represented as evidence that earlier overfitting is resolved.


In [ ]:
feature_tables = {}
for strategy in ('filter', 'wrapper', 'embedded'):
    table = selection_result[strategy].merge(feature_lineage.rename(columns={'FEATURE': 'FEATURE_NAME'}),
                                             on='FEATURE_NAME', how='left', validate='one_to_one')
    require(len(table) == X.shape[1] and table.SOURCE_TABLE.notna().all(), 'Incomplete ranking lineage.')
    table['RUN_ID'] = RUN_ID
    table['INPUT_SHA256'] = input_fingerprint
    table['IMPLEMENTATION_SHA256'] = IMPLEMENTATION_SHA256
    feature_tables[strategy] = table
    print(strategy.upper(), '| selected:', int(table.SELECTED.sum()), '| audited:', len(table))
    display(table)
filter_features = feature_tables['filter']
wrapper_features = feature_tables['wrapper']
embedded_features = feature_tables['embedded']


## 11. Visualize comparison, selected-feature overlap and stability

**Why:** fewer inputs help only if out-of-fold usefulness and stability remain acceptable. Compare AP
and lift with `all_eligible`, then inspect overlap. Disagreement can arise from correlated substitutes,
interactions or sampling variability; it does not establish that one method is wrong.

**Outputs:** mean TRAIN-fold AP/lift with fold standard deviations, pairwise selected-set overlap,
and top selected-feature stability bars. Error bars are descriptive fold variability, not confidence
intervals. If a subset performs worse, do not call feature reduction an improvement. With three folds,
stability has only four possible values (0, 1/3, 2/3, 1), so interpret it cautiously.


In [ ]:
comparison = diagnostics.groupby('STRATEGY').agg(
    AP_MEAN=('AVERAGE_PRECISION', 'mean'), AP_FOLD_SD=('AVERAGE_PRECISION', 'std'),
    LIFT_MEAN=('TOP10_LIFT', 'mean'), LIFT_FOLD_SD=('TOP10_LIFT', 'std'),
    SELECTED_MEAN=('SELECTED_FEATURES', 'mean')).reset_index()
display(comparison)
chosen = {method: set(table.loc[table.SELECTED, 'FEATURE_NAME']) for method, table in feature_tables.items()}
overlap = pd.DataFrame({a: {b: len(chosen[a] & chosen[b]) for b in chosen} for a in chosen})
display(overlap.rename_axis('SELECTED_SET'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
axes[0].bar(comparison.STRATEGY, comparison.AP_MEAN, yerr=comparison.AP_FOLD_SD.fillna(0), capsize=4)
axes[0].axhline(y.mean(), color='grey', linestyle='--', label='Analysis positive prevalence')
axes[0].set(title='TRAIN-fold AP: fixed L2-logistic evaluator', ylabel='Average precision')
axes[0].legend()
axes[1].bar(comparison.STRATEGY, comparison.LIFT_MEAN, yerr=comparison.LIFT_FOLD_SD.fillna(0), capsize=4)
axes[1].axhline(1, color='grey', linestyle='--')
axes[1].set(title='TRAIN-fold top-10% lift', ylabel='Lift')
plt.show()
for method, table in feature_tables.items():
    top = table.loc[table.SELECTED].head(15).iloc[::-1]
    if top.empty:
        print(method, ': no features selected; inspect quality/selection reasons.')
        continue
    fig, ax = plt.subplots(figsize=(13, max(4, 0.32 * len(top))), layout='constrained')
    ax.barh(top.FEATURE_NAME, top.FOLD_SELECTION_FREQUENCY)
    ax.set(xlim=(0, 1.05), xlabel='Fraction of TRAIN folds selecting feature',
           title=method.upper() + ': stability of highest-ranked selected features')
    plt.show()


## 12. Save three new Snowflake feature tables and verify read-back

**Why:** downstream experiments need actual selected lists and audit rows, not a screenshot of top bars.
This cell creates `{OUTPUT_PREFIX}_FILTER`, `_WRAPPER`, `_EMBEDDED`. Each holds the full candidate audit,
selected flags, lineage, input fingerprints, settings, runtime versions and aggregate fold diagnostics.
No patient IDs, per-patient values or predictions are written. Tables are private project artifacts.

**Output:** exact fully qualified table names, verified row counts and selected-feature counts. Creation
uses `errorifexists`; all three names are preflighted before writing. If a connector error interrupts
saving, partial new tables may exist. Rerun with a fresh `RUN_ID`; the code never deletes source/results.
Save/read-back verification compares feature names, ranks and selected flags. `WRITE_RESULTS=False`
keeps the same three tables as DataFrames in the session and performs no warehouse writes.

**Next experiment:** freeze these definitions, compare the three lists with the full-feature control
in the intended downstream model using equal tuning budgets, and evaluate on a separately reserved
population. Do not reuse a global preselected list inside supposedly independent TRAIN cross-validation;
re-fit the selector within each training fold, as implemented here.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType

run_manifest = {
    'run_id': RUN_ID, 'input_sha256': input_fingerprint,
    'implementation_sha256': IMPLEMENTATION_SHA256,
    'source_tables': TABLES, 'sum_windows': list(SUM_WINDOWS),
    'extra_source_contracts': EXTRA_SOURCE_CONTRACTS,
    'selection_config': selection_result['config'],
    'analysis_unit': 'latest snapshot per frozen TRAIN patient',
    'analysis_patients': len(train_metadata), 'analysis_positives': int(y.sum()),
    'candidate_count': X.shape[1],
    'upstream_limitations': ['Original claim category/deduplication SQL not recovered',
                            'Original event/ingestion cutoff and label construction not independently verified'],
    'diagnostics': diagnostics.to_dict('records'),
    'versions': {'numpy': np.__version__, 'pandas': pd.__version__,
                 'scipy': scipy.__version__, 'sklearn': sklearn.__version__},
}
run_manifest_json = json.dumps(run_manifest, sort_keys=True, allow_nan=False)
destinations = {method: OUTPUT_PREFIX + '_' + method.upper() for method in feature_tables}

def to_spark_report(frame):
    fields, converters = [], []
    for column in frame.columns:
        dtype = frame[column].dtype
        if pd.api.types.is_bool_dtype(dtype):
            kind, convert = BooleanType(), bool
        elif pd.api.types.is_integer_dtype(dtype):
            kind, convert = LongType(), int
        elif pd.api.types.is_numeric_dtype(dtype):
            kind, convert = DoubleType(), float
        else:
            kind, convert = StringType(), str
        fields.append(StructField(str(column), kind, True))
        converters.append(convert)
    records = [tuple(None if pd.isna(value) else convert(value)
                     for value, convert in zip(row, converters))
               for row in frame.itertuples(index=False, name=None)]
    return spark.createDataFrame(records, StructType(fields))

saved_results = []
if WRITE_RESULTS:
    for destination in destinations.values():
        database, schema, table_name = destination.split('.')
        qtable(destination)
        existing = read_sql(f'SELECT TABLE_NAME FROM {qi(database)}.INFORMATION_SCHEMA.TABLES '
                            f'WHERE TABLE_SCHEMA = {literal(schema)} AND TABLE_NAME = {literal(table_name)}')
        require(existing.empty, 'Output already exists; use a fresh RUN_ID: ' + destination)
    for method, destination in destinations.items():
        exported = feature_tables[method].copy()
        exported['RUN_MANIFEST_JSON'] = run_manifest_json
        (to_spark_report(exported).write.format('snowflake').options(**sf_options_analysis)
         .option('dbtable', destination).mode('errorifexists').save())
        observed = read_sql('SELECT FEATURE_NAME, RANK, SELECTED, RUN_ID FROM ' + qtable(destination))
        expected = exported[['FEATURE_NAME', 'RANK', 'SELECTED', 'RUN_ID']].sort_values('FEATURE_NAME').reset_index(drop=True)
        observed = observed.sort_values('FEATURE_NAME').reset_index(drop=True)
        pd.testing.assert_frame_equal(expected, observed, check_dtype=False)
        saved_results.append({'STRATEGY': method, 'TABLE': destination, 'ROWS_VERIFIED': len(observed),
                              'SELECTED_FEATURES': int(exported.SELECTED.sum()), 'STATUS': 'SAVED_AND_VERIFIED'})
else:
    saved_results = [{'STRATEGY': method, 'TABLE': destination, 'ROWS_VERIFIED': len(feature_tables[method]),
                      'SELECTED_FEATURES': int(feature_tables[method].SELECTED.sum()),
                      'STATUS': 'DATAFRAME_ONLY_NOT_WRITTEN'} for method, destination in destinations.items()]
display(pd.DataFrame(saved_results))


## References and scope

- [scikit-learn feature selection](https://scikit-learn.org/stable/modules/feature_selection.html): filter, RFE and embedded methods.
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html): classification penalties and regularization controls.
- [Common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html): fit preprocessing/selection only on training data.

This is a feature discovery and selection notebook, not a deployment or a reproduction of previous
model results. Table availability and real rankings are established only when it runs in your private
environment. The committed copy has cleared outputs and contains no credentials or patient data.
